# EDA - Dataset de Churn (Base de Treino)

Analise exploratoria do dataset agregado por VIN (`dataset_churn_pos_venda.csv`), usado para o treinamento dos modelos de Churn e Perfil.

In [9]:

import matplotlib
matplotlib.use('Agg')

import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Fix CWD
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    os.chdir(notebook_dir.parent)
print(f"CWD: {os.getcwd()}")

os.makedirs('reports', exist_ok=True)
sns.set_theme(style='whitegrid')

CWD: /Users/nando_mac/Desktop/Ford_Dock 360


## Carregamento e visao geral

In [10]:
base_path = 'data/processed/dataset_churn_pos_venda.csv'
if not os.path.exists(base_path):
    print(f"AVISO: {base_path} nao encontrado. Rode o pipeline primeiro.")
else:
    df = pd.read_csv(base_path)
    print(f'Linhas: {df.shape[0]:,}')
    print(f'Colunas: {df.shape[1]:,}')
    display(df.head())

Linhas: 118,379
Colunas: 29


,VIN_Hash,qtde_revisoes_ate_corte,primeiro_servico_ate_corte,ultimo_servico_ate_corte,modelo,ano_modelo,dealer_code_ate_corte,n_dealers_usados_ate_corte,km_max_ate_corte,sales_date,...,intervalo_medio_revisoes_dias_ate_corte,qtd_servicos_pos_corte,primeiro_servico_pos_corte,ultimo_servico_pos_corte,voltou_pos_corte,churn_futuro_18m,data_corte,fim_janela_churn,data_max_observada,janela_futura_observavel
0,00008eef200a0a71fd52b4b0bd43c6e275b50a8fe56a6f...,1,2022-03-03,2022-03-03,KA,2021.0,3050,1,11010.0,2021-01-22,...,NaN,0,NaN,NaN,False,1,2024-10-31,2026-04-30,2026-05-04,True
1,00015a2ad3fd2a788fb2ccf83b443e5326b3d835b53468...,1,2020-11-16,2020-11-16,RANGER,2020.0,2606,1,9462.0,2020-02-10,...,NaN,0,NaN,NaN,False,1,2024-10-31,2026-04-30,2026-05-04,True
2,0002a2d7b0e64c34a5cd3fdc69c387d28128474230d897...,3,2022-04-13,2023-09-21,KA,2021.0,5650,1,50605.0,2020-11-09,...,263.0,0,NaN,NaN,False,1,2024-10-31,2026-04-30,2026-05-04,True
3,00031d6a4d99a0ccf6222ea9beb894e562ec5749603d92...,3,2022-01-10,2024-01-25,ECOSPORT,2021.0,6163,1,30251.0,2021-01-15,...,372.5,1,2025-01-31,2025-01-31,True,0,2024-10-31,2026-04-30,2026-05-04,True
4,00033ec3f308cce9068684af00cfa6072fed00f9567989...,3,2021-11-19,2022-10-17,KA,2021.0,6137,2,32473.0,2020-12-23,...,166.0,0,NaN,NaN,False,1,2024-10-31,2026-04-30,2026-05-04,True


## Distribuicao de Churn (Real)

In [11]:
from src.pipeline.config import TARGET_CHURN

if TARGET_CHURN in df.columns:
    plt.figure(figsize=(7, 5))
    churn_counts = df[TARGET_CHURN].value_counts(normalize=True).sort_index().mul(100)
    sns.barplot(x=churn_counts.index, y=churn_counts.values, hue=churn_counts.index, palette='RdYlGn_r', legend=False)
    plt.title('Distribuicao de Churn Futuro 18m')
    plt.ylabel('Percentual (%)')
    plt.xlabel('Classe')
    plt.xticks([0, 1], ['No Churn', 'Churn'])
    for i, v in enumerate(churn_counts.values):
        plt.text(i, v + 0.5, f'{v:.1f}%', ha='center')
    plt.savefig('reports/base2_distribuicao_churn.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Coluna target ausente: {TARGET_CHURN}')


/var/folders/_s/ht420jlx6vj17yhfdml8dr2r0000gn/T/ipykernel_300/19428481.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Correlacao de Features Comportamentais com o Target

In [12]:
from src.pipeline.config import SNAPSHOT_FEATURES_NUMERIC, TARGET_CHURN

behav_cols = [c for c in SNAPSHOT_FEATURES_NUMERIC if c in df.columns]
corr_cols = behav_cols + [TARGET_CHURN]
corr = df[corr_cols].corr(numeric_only=True)[TARGET_CHURN].drop(TARGET_CHURN).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette='coolwarm', legend=False)
plt.title('Correlacao de Features Comportamentais com Churn Futuro 18m')
plt.xlabel('Coeficiente de Correlacao')
plt.savefig('reports/base2_correlacao_churn.png', dpi=150, bbox_inches='tight')
plt.show()


/var/folders/_s/ht420jlx6vj17yhfdml8dr2r0000gn/T/ipykernel_300/732654246.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
